# AI-Generated A/B Test Analysis: Validation & Debugging

## Purpose
This notebook validates an AI-generated analysis of an A/B test.

## Step 1: The AI's Original Analysis

Below is the AI's generated code and analysis, preserved exactly as provided.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv('../data/ab_test_data.csv')

control_rate = df[df['group'] == 'control']['converted'].mean()
treatment_rate = df[df['group'] == 'treatment']['converted'].mean()

print(f"Control conversion rate: {control_rate:.2%}")
print(f"Treatment conversion rate: {treatment_rate:.2%}")
print(f"Lift: {((treatment_rate - control_rate) / control_rate):.2%}")

control_converted = df[df['group'] == 'control']['converted']
treatment_converted = df[df['group'] == 'treatment']['converted']

t_stat, p_value = stats.ttest_ind(control_converted, treatment_converted)

print(f"\nT-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("\n✓ Result is statistically significant (p < 0.05)")
    print("✓ The treatment is better than control")
else:
    print("\n✗ Result is not statistically significant")
    print("✗ No evidence that treatment works")

print("\n\n### RECOMMENDATION ###")
print("Based on the analysis, we recommend implementing the treatment")
print("as it shows a statistically significant improvement in conversions.")

## Step 2: Initial Validation/what Looks Good?

**First impressions:**
- ✓ Code runs without errors
- ✓ Calculates basic metrics (conversion rates, lift)
- ✓ Uses appropriate test for binary outcome comparison
- ✓ Prints clear output

**Must find if its correct**

## Step 3: My Validation - Identifying Issues

Ill run the code and systematically check for errors in three areas:
1. **Statistical methodology** (wrong test, assumption violations)
2. **Code correctness** (bugs, logical errors)
3. **Interpretation** (misunderstanding results)

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.proportion import proportions_ztest

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

df = pd.read_csv('../data/ab_test_data.csv')

print("=== DATA SUMMARY ===")
print(f"Total observations: {len(df):,}")
print(f"Control group: {len(df[df['group'] == 'control']):,}")
print(f"Treatment group: {len(df[df['group'] == 'treatment']):,}")
print(f"\nSample characteristics:")
print(df.describe())

In [ ]:
control_rate = df[df['group'] == 'control']['converted'].mean()
treatment_rate = df[df['group'] == 'treatment']['converted'].mean()

print("=== AI'S RESULTS (EXECUTED) ===")
print(f"Control conversion rate: {control_rate:.2%}")
print(f"Treatment conversion rate: {treatment_rate:.2%}")
print(f"Lift: {((treatment_rate - control_rate) / control_rate):.2%}")

control_converted = df[df['group'] == 'control']['converted']
treatment_converted = df[df['group'] == 'treatment']['converted']

t_stat, p_value = stats.ttest_ind(control_converted, treatment_converted)

print(f"\nT-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("\n✓ AI says: Result is statstically significant")
    print("✓ AI says: Treatment is better than control")
else:
    print("\n✗ Result is not statistically significant")

## Step 4: Critical Issue #1 - Statistical Method

### The Problem
The AI used a t-test for a binary outcome (converted = 0 or 1).

**Why this is wrong:**
- T-test assumes normally distributed data
- Binary data is not normally distributed
- For proportions, we should use a proportion test (z-test or chi-square)

**Severity:** MEDIUM - The conclusion may still be valid howeevr methodology is technically incorrect.

In [ ]:
control_successes = df[df['group'] == 'control']['converted'].sum()
treatment_successes = df[df['group'] == 'treatment']['converted'].sum()
control_n = len(df[df['group'] == 'control'])
treatment_n = len(df[df['group'] == 'treatment'])

print("=== CORRECT STATISTICAL TEST: PROPORTION Z-TEST ===")
print(f"Control: {control_successes} conversions out of {control_n}")
print(f"Treatment: {treatment_successes} conversions out of {treatment_n}")

count = np.array([control_successes, treatment_successes])
nobs = np.array([control_n, treatment_n])
z_stat, p_value_correct = proportions_ztest(count, nobs, alternative='two-sided')

print(f"\nZ-statistic: {z_stat:.4f}")
print(f"P-value (proportion test): {p_value_correct:.4f}")
print(f"\nAI's t-test p-value: {p_value:.4f}")
print(f"Difference: {abs(p_value - p_value_correct):.4f}")

if p_value_correct < 0.05:
    print("\n✓ Correct test also shows significance")
    print("  → AI's conclusion was right but method was wrong")
else:
    print("\n⚠ AI's conclusion may be wrong")
    print("  → This is dangerous because incorrect method could lead to false positives")

## Step 5: Critical Issue #2 , Missing Assumption Checks

### The Problem
The AI didn't check any assumptions:
1. Independence , Are observations independent?
2. Sample size , Is it large enough for the test?
3. Random assignment , Was it truly randomized?

**Severity:** HIGH - Unchecked assumptions can invalidate the entire analysis

In [ ]:
print("=== ASSUMPTION CHECKS ===\n")

print("1. Sample Size Adequacy:")
min_successes = min(control_successes, treatment_successes)
min_failures = min(control_n - control_successes, treatment_n - treatment_successes)

print(f"   Smallest success count: {min_successes}")
print(f"   Smallest failure count: {min_failures}")

if min_successes >= 10 and min_failures >= 10:
    print("    Sample size is adequate (all cells >= 10)")
else:
    print("    Sample size may be too small - consider Fisher's exact test")

print("\n2. Random Assignment Check:")
if 'pre_test_score' in df.columns:
    control_score = df[df['group'] == 'control']['pre_test_score'].mean()
    treatment_score = df[df['group'] == 'treatment']['pre_test_score'].mean()
    print(f"   Control mean pre-test: {control_score:.2f}")
    print(f"   Treatment mean pre-test: {treatment_score:.2f}")
    
    _, p_balance = stats.ttest_ind(
        df[df['group'] == 'control']['pre_test_score'],
        df[df['group'] == 'treatment']['pre_test_score']
    )
    if p_balance > 0.05:
        print(f"    Groups appear balanced (p={p_balance:.3f})")
    else:
        print(f"    Groups may not be balanced (p={p_balance:.3f})")

print("\n3. Effect Size:")
effect_size = treatment_rate - control_rate
relative_lift = (treatment_rate - control_rate) / control_rate
print(f"   Absolute difference: {effect_size:.2%}")
print(f"   Relative lift: {relative_lift:.2%}")

import math
h = 2 * math.asin(math.sqrt(treatment_rate)) - 2 * math.asin(math.sqrt(control_rate))
print(f"   Cohen's h: {h:.3f}")
if abs(h) < 0.2:
    print("   → Small effect size")
elif abs(h) < 0.5:
    print("   → Medium effect size")
else:
    print("   → Large effect size")

## Step 6: Critical Issue #3 - Interpretation Errors

### The Problem
The AI made a serious **statistical interpretation error**:

**AI's statement:** *"p-value < 0.05, so the treatment is better than control"*

**This is WRONG.** The p-value is NOT the probability that the treatment is better.

**Correct interpretation:** Given the null hypothesis (no difference), there's a {p-value} probability of observing a difference this large or larger.

**Severity:** HIGH.. This is a fundamental misunderstanding that leads to overconfident conclusions.

In [ ]:
print("=== INTERPRETATION VALIDATION ===\n")

print("What the AI said:")
print('  "p-value < 0.05 so the treatment is better than control"')
print()
print("What the p-value actually means:")
print(f"  P(data | null hypothesis is true) = {p_value_correct:.4f}")
print()
print("Correct interpretation:")
print(f"  If there were truly no difference between groups,")
print(f"  we would observe a difference this large or larger")
print(f"  only {p_value_correct:.2%} of the time.")
print()
print(" This is a classic and dangerous mistake.")
print("  The AI conclusion may be correct, but the reasoning is flawed.")
print("  In a business context this could lead to overconfidence in results.")

## Step 7: Critical Issue #4 - Business Recommendations

### The Problem
The AI's recommendation was too simplistic and missing:
- Cost-benefit analysis
- Confidence intervals
- Implementation complexity
- Practical significance

**Severity:** MEDIUM - The recommendation might be fine but it lacks nuance.

In [ ]:
print("=== BUSINESS RECOMMENDATION ANALYSIS ===\n")

def proportion_ci(successes, n, confidence=0.95):
    p = successes / n
    se = math.sqrt(p * (1-p) / n)
    z = 1.96
    return p - z*se, p + z*se

control_ci = proportion_ci(control_successes, control_n)
treatment_ci = proportion_ci(treatment_successes, treatment_n)

print("95% Confidence Intervals:")
print(f"  Control: [{control_ci[0]:.2%}, {control_ci[1]:.2%}]")
print(f"  Treatment: [{treatment_ci[0]:.2%}, {treatment_ci[1]:.2%}]")

difference = treatment_rate - control_rate
total_users = len(df)
expected_extra_conversions = difference * total_users

print(f"\nEstimated additional conversions if rolled out to all users:")
print(f"  {expected_extra_conversions:.0f} extra conversions")

print("\nWhat a good recommendation would include:")
print("  1. Statistical findings (with CI)")
print("  2. Cost-benefit analysis")
print("  3. Implementation considerations")
print("  4. Potential risks")
print("  5. Next steps (e.g., holdout validation)")